In [6]:
%load_ext autoreload
%autoreload 2

from pyspark.sql import SparkSession

# -------------------------------
# 1. Spark Session
# -------------------------------
spark = SparkSession.builder \
    .appName("Decision_engine") \
    .master("spark://localhost:7077") \
    .config("spark.driver.host", "172.21.0.1") \
    .config("spark.driver.bindAddress", "0.0.0.0") \
    .config("spark.driver.port", "41251") \
    .config("spark.blockManager.port", "41252") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

spark

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
from pyspark.sql.functions import col, from_json, explode
from pyspark.sql.types import *
import json
from decision_engine import decide_activity

schema = ArrayType(StructType([
    StructField("context_id", StringType()),
    StructField("user_id", StringType()),
    StructField("created_at", StringType()),

    StructField("vision", StructType([
        StructField("timestamp", StringType()),
        StructField("objects", ArrayType(StringType())),
        StructField("scene_description", StringType()),
        StructField("confidence", DoubleType()),
        StructField("media_ref", StringType())
    ])),

    StructField("audio", StructType([
        StructField("timestamp", StringType()),
        StructField("transcript", StringType()),
        StructField("keywords", ArrayType(StringType())),
        StructField("confidence", DoubleType()),
        StructField("audio_ref", StringType())
    ])),

    StructField("location", StructType([
        StructField("timestamp", StringType()),
        StructField("latitude", DoubleType()),
        StructField("longitude", DoubleType()),
        StructField("place_label", StringType()),
        StructField("zone_type", StringType())
    ]))
]))

df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "contextBuilder") \
    .option("startingOffsets", "earliest") \
    .load()

parsed_df = df.selectExpr("CAST(value AS STRING) as value") \
    .select(from_json(col("value"), schema).alias("data")) \
    .select(explode(col("data")).alias("context"))



In [8]:
def process_batch(batch_df, batch_id):
    print(f"\n--- batch_id={batch_id} ---")
    rows = batch_df.collect()
    print(f"rows={len(rows)}")

    for i, row in enumerate(rows, 1):
        try:
            context = row["context"].asDict(recursive=True)
            print(f"processing row {i} / {len(rows)} | context_id={context.get('context_id')}")
            decision = decide_activity(context)
            print(json.dumps(decision, indent=2, ensure_ascii=False))
        except Exception as e:
            print(f"ERROR on row {i}: {e}")
            raise

query = parsed_df.writeStream \
    .foreachBatch(process_batch) \
    .outputMode("append") \
    .start()

print(query.status)


{'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


26/04/26 21:34:49 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-e724599e-c314-4189-85bb-2f9d9dbe97d2. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/26 21:34:49 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [9]:
print("isActive:", query.isActive)
print("status:", query.status)
print("recentProgress:", query.recentProgress[-1] if query.recentProgress else "no progress yet")
print("exception:", query.exception())


isActive: True
status: {'message': 'Getting offsets from KafkaV2[Subscribe[contextBuilder]]', 'isDataAvailable': False, 'isTriggerActive': True}
recentProgress: no progress yet
exception: None


In [10]:
query.awaitTermination(20)


26/04/26 21:34:49 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


False

In [11]:
spark.stop()